In [ ]:
import pandas as pd
import numpy as np

elec = pd.read_csv("data_raw/electricity/eurostat_electricity_prices_2023_2025.csv")

elec = elec[["geo", "TIME_PERIOD", "OBS_VALUE"]]

selected_countries = [
    "United Kingdom",
    "Ireland",
    "Germany",
    "Netherlands",
    "France",
    "Spain",
    "Italy",
    "Belgium",
    "Poland",
    "Austria",
    "Denmark",
    "Sweden",
    "Finland",
    "Norway",
    "Switzerland"
]

elec = elec[elec["geo"].isin(selected_countries)]

elec = elec[elec["TIME_PERIOD"].isin([2022, 2023, 2024])]

elec["OBS_VALUE"] = pd.to_numeric(elec["OBS_VALUE"], errors="coerce")

elec_avg = (
    elec.groupby("geo")["OBS_VALUE"]
    .mean()
    .reset_index()
    .rename(columns={"OBS_VALUE": "elec_price_eur_kwh"})
)

elec_avg

,geo,elec_price_eur_kwh
0,Austria,0.035041
1,Belgium,0.042641
2,Denmark,0.038308
3,Finland,0.018930
4,France,0.037833
5,Germany,0.047116
6,Ireland,0.040593
7,Italy,0.048737
8,Netherlands,0.045444
9,Norway,0.023817


In [2]:
elec_avg.shape
elec_avg


,geo,elec_price_eur_kwh
0,Austria,0.035041
1,Belgium,0.042641
2,Denmark,0.038308
3,Finland,0.018930
4,France,0.037833
5,Germany,0.047116
6,Ireland,0.040593
7,Italy,0.048737
8,Netherlands,0.045444
9,Norway,0.023817


In [3]:
sorted(elec["geo"].unique())


['Austria',
 'Belgium',
 'Denmark',
 'Finland',
 'France',
 'Germany',
 'Ireland',
 'Italy',
 'Netherlands',
 'Norway',
 'Poland',
 'Spain',
 'Sweden']

In [4]:
countries = [
    "Ireland",
    "Germany",
    "Netherlands",
    "France",
    "Spain",
    "Italy",
    "Belgium",
    "Poland",
    "Austria",
    "Denmark",
    "Sweden",
    "Finland",
    "Norway",
]

df_master = pd.DataFrame({"country": countries})

df_master


,country
0,Ireland
1,Germany
2,Netherlands
3,France
4,Spain
5,Italy
6,Belgium
7,Poland
8,Austria
9,Denmark


In [5]:
df_master = df_master.merge(
    elec_avg,
    left_on="country",
    right_on="geo",
    how="left"
).drop(columns=["geo"])

df_master


,country,elec_price_eur_kwh
0,Ireland,0.040593
1,Germany,0.047116
2,Netherlands,0.045444
3,France,0.037833
4,Spain,0.032659
5,Italy,0.048737
6,Belgium,0.042641
7,Poland,0.045631
8,Austria,0.035041
9,Denmark,0.038308


In [6]:
ember = pd.read_csv("data_raw/carbon/ember_yearly_electricity_data.csv")

ember.head()


,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Clean,GW,0.19,NaN,NaN
1,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Fossil,GW,0.03,NaN,NaN
2,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Gas and Other Fossil,GW,0.03,NaN,NaN
3,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,"Hydro, Bioenergy and Other Renewables",GW,0.19,NaN,NaN
4,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Renewables,GW,0.19,NaN,NaN


In [7]:
ember.columns


Index(['Area', 'ISO 3 code', 'Year', 'Area type', 'Continent', 'Ember region',
       'EU', 'OECD', 'G20', 'G7', 'ASEAN', 'Category', 'Subcategory',
       'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change'],
      dtype='str')

In [8]:
sorted(ember["Variable"].unique())


['Bioenergy',
 'CO2 intensity',
 'Clean',
 'Coal',
 'Demand',
 'Demand per capita',
 'Fossil',
 'Gas',
 'Gas and Other Fossil',
 'Hydro',
 'Hydro, Bioenergy and Other Renewables',
 'Net Imports',
 'Nuclear',
 'Other Fossil',
 'Other Renewables',
 'Renewables',
 'Solar',
 'Total Generation',
 'Total emissions',
 'Wind',
 'Wind and Solar']

In [ ]:
ember = pd.read_csv("data_raw/carbon/ember_yearly_electricity_data.csv")

selected_countries = df_master["country"].tolist()

ember = ember[ember["Area"].isin(selected_countries)]

ember = ember[ember["Year"].isin([2022, 2023, 2024])]


In [10]:
carbon = ember[ember["Variable"] == "CO2 intensity"]

carbon_avg = (
    carbon.groupby("Area")["Value"]
    .mean()
    .reset_index()
    .rename(columns={"Area": "country",
                     "Value": "carbon_intensity_gco2_kwh"})
)

carbon_avg


,country,carbon_intensity_gco2_kwh
0,Austria,118.426667
1,Belgium,133.650000
2,Denmark,162.113333
3,Finland,92.830000
4,France,57.543333
5,Germany,374.376667
6,Ireland,296.400000
7,Italy,327.526667
8,Netherlands,281.423333
9,Norway,30.290000


In [11]:
renew = ember[ember["Variable"] == "Renewables"]

renew_avg = (
    renew.groupby("Area")["Value"]
    .mean()
    .reset_index()
    .rename(columns={"Area": "country",
                     "Value": "renew_share_pct"})
)

renew_avg


,country,renew_share_pct
0,Austria,42.835833
1,Belgium,18.480000
2,Denmark,32.487500
3,Finland,28.236667
4,France,58.116667
5,Germany,125.048333
6,Ireland,15.863333
7,Italy,58.158333
8,Netherlands,34.255833
9,Norway,72.872500


In [12]:
df_master = df_master.merge(carbon_avg, on="country", how="left")
df_master = df_master.merge(renew_avg, on="country", how="left")

df_master


,country,elec_price_eur_kwh,carbon_intensity_gco2_kwh,renew_share_pct
0,Ireland,0.040593,296.400000,15.863333
1,Germany,0.047116,374.376667,125.048333
2,Netherlands,0.045444,281.423333,34.255833
3,France,0.037833,57.543333,58.116667
4,Spain,0.032659,177.430000,69.998333
5,Italy,0.048737,327.526667,58.158333
6,Belgium,0.042641,133.650000,18.480000
7,Poland,0.045631,666.380000,25.310000
8,Austria,0.035041,118.426667,42.835833
9,Denmark,0.038308,162.113333,32.487500


In [15]:
temp = pd.read_csv("data_raw/temperature/worldbank_avg_annual_temperature_2022_2024.csv")

temp.head()


,code,name,1901-07,1902-07,1903-07,1904-07,1905-07,1906-07,1907-07,1908-07,...,2015-07,2016-07,2017-07,2018-07,2019-07,2020-07,2021-07,2022-07,2023-07,2024-07
0,ABW,Aruba (Neth.),28.22,27.79,27.89,27.62,27.68,27.58,27.56,27.46,...,29.67,29.68,29.27,28.97,29.37,29.41,29.06,28.88,29.45,29.63
1,AFG,Afghanistan,12.78,12.98,11.81,12.13,12.02,12.50,11.89,12.21,...,13.75,14.27,13.86,14.13,13.81,13.15,14.31,14.46,14.61,14.24
2,AGO,Angola,21.32,21.36,21.37,21.28,21.32,21.20,21.22,21.29,...,21.70,21.94,21.82,21.66,21.81,21.73,21.73,21.62,21.56,21.71
3,AIA,Anguilla (U.K.),27.18,26.77,26.80,26.33,26.69,26.44,26.10,26.24,...,28.23,28.16,27.96,27.69,28.08,28.10,27.97,27.82,28.29,28.72
4,ALA,ALA,5.39,3.43,5.29,4.72,5.17,5.70,4.55,4.88,...,6.87,6.36,6.18,6.94,6.95,7.78,6.27,6.92,6.66,7.30


In [ ]:
import pandas as pd
import numpy as np

temp = pd.read_csv("data_raw/temperature/worldbank_avg_annual_temperature_2022_2024.csv")

[c for c in temp.columns if c.startswith(("2022", "2023", "2024"))][:20]


['2022-07', '2023-07', '2024-07']

In [ ]:
year_cols = ["2022-07", "2023-07", "2024-07"]

temp_small = temp[temp["name"].isin(df_master["country"])].copy()

temp_small[year_cols] = temp_small[year_cols].apply(pd.to_numeric, errors="coerce")

temp_small["avg_temp_c"] = temp_small[year_cols].mean(axis=1)

temp_avg = temp_small[["name", "avg_temp_c"]].rename(columns={"name": "country"})

temp_avg.sort_values("country")


,country,avg_temp_c
14,Austria,8.916667
17,Belgium,11.890000
61,Denmark,9.770000
70,Finland,3.243333
72,France,12.950000
58,Germany,10.956667
103,Ireland,10.463333
108,Italy,14.476667
164,Netherlands,11.706667
165,Norway,2.766667


In [19]:
df_master = df_master.merge(temp_avg, on="country", how="left")
df_master


,country,elec_price_eur_kwh,carbon_intensity_gco2_kwh,renew_share_pct,avg_temp_c
0,Ireland,0.040593,296.400000,15.863333,10.463333
1,Germany,0.047116,374.376667,125.048333,10.956667
2,Netherlands,0.045444,281.423333,34.255833,11.706667
3,France,0.037833,57.543333,58.116667,12.950000
4,Spain,0.032659,177.430000,69.998333,15.140000
5,Italy,0.048737,327.526667,58.158333,14.476667
6,Belgium,0.042641,133.650000,18.480000,11.890000
7,Poland,0.045631,666.380000,25.310000,10.300000
8,Austria,0.035041,118.426667,42.835833,8.916667
9,Denmark,0.038308,162.113333,32.487500,9.770000


In [20]:
df_master["avg_temp_c"].describe()
df_master[df_master["avg_temp_c"].isna()]


,country,elec_price_eur_kwh,carbon_intensity_gco2_kwh,renew_share_pct,avg_temp_c


In [21]:
loss = pd.read_csv("data_raw/reliability/worldbank_transmission_losses.csv")
loss.head()
loss.columns


ParserError: Error tokenizing data. C error: Expected 3 fields in line 5, saw 70


In [23]:
with open("data_raw/reliability/worldbank_transmission_losses.csv", "r", encoding="utf-8", errors="ignore") as f:
    for i in range(1, 15):
        print(i, f.readline().strip()[:200])


1 ﻿"Data Source","World Development Indicators",
2 
3 "Last Updated Date","2026-01-28",
4 
5 "Country Name","Country Code","Indicator Name","Indicator Code","1960","1961","1962","1963","1964","1965","1966","1967","1968","1969","1970","1971","1972","1973","1974","1975","1976","1977","1978","19
6 "Aruba","ABW","Electric power transmission and distribution losses (% of output)","EG.ELC.LOSS.ZS","","","","","","","","","","","","","","","","","","","","","","","","","","","","","","","","","",""
7 "Africa Eastern and Southern","AFE","Electric power transmission and distribution losses (% of output)","EG.ELC.LOSS.ZS","","","","","","","","","","","","","","","","","","","","","","","","","","","
8 "Afghanistan","AFG","Electric power transmission and distribution losses (% of output)","EG.ELC.LOSS.ZS","","","","","","","","","","","","","","","","","","","","","","","","","","","","","","","",""
9 "Africa Western and Central","AFW","Electric power transmission and distribution losses (% of o

In [24]:
loss = pd.read_csv(
    "data_raw/reliability/worldbank_transmission_losses.csv",
    skiprows=4
)

loss.head(), loss.columns


(                  Country Name Country Code  \
 0                        Aruba          ABW   
 1  Africa Eastern and Southern          AFE   
 2                  Afghanistan          AFG   
 3   Africa Western and Central          AFW   
 4                       Angola          AGO   
 
                                       Indicator Name  Indicator Code  1960  \
 0  Electric power transmission and distribution l...  EG.ELC.LOSS.ZS   NaN   
 1  Electric power transmission and distribution l...  EG.ELC.LOSS.ZS   NaN   
 2  Electric power transmission and distribution l...  EG.ELC.LOSS.ZS   NaN   
 3  Electric power transmission and distribution l...  EG.ELC.LOSS.ZS   NaN   
 4  Electric power transmission and distribution l...  EG.ELC.LOSS.ZS   NaN   
 
    1961  1962  1963  1964  1965  ...       2016       2017       2018  \
 0   NaN   NaN   NaN   NaN   NaN  ...        NaN        NaN        NaN   
 1   NaN   NaN   NaN   NaN   NaN  ...  11.070454  10.831054  11.312177   
 2   NaN   N

In [25]:
loss = pd.read_csv(
    "data_raw/reliability/worldbank_transmission_losses.csv",
    engine="python",
    sep=",",
    on_bad_lines="skip"
)

loss.head(), loss.columns


(         Data Source World Development Indicators  Unnamed: 2
 0  Last Updated Date                   2026-01-28         NaN,
 Index(['Data Source', 'World Development Indicators', 'Unnamed: 2'], dtype='str'))

In [26]:
loss_small = loss[loss["Country Name"].isin(df_master["country"])].copy()

year_cols = ["2022", "2023", "2024"]
loss_small[year_cols] = loss_small[year_cols].apply(pd.to_numeric, errors="coerce")

loss_small["t_d_losses_pct"] = loss_small[year_cols].mean(axis=1)

loss_avg = loss_small[["Country Name", "t_d_losses_pct"]].rename(columns={"Country Name": "country"})

df_master = df_master.merge(loss_avg, on="country", how="left")
df_master


KeyError: 'Country Name'

In [29]:
import csv

path = "data_raw/reliability/worldbank_transmission_losses.csv"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if line.startswith("Country Name") or line.startswith('"Country Name"'):
            print("Header row found at line:", i)
            break


Header row found at line: 4


In [30]:
loss = pd.read_csv(path, skiprows=4)
loss.head()
loss.columns[:10]


Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965'],
      dtype='str')

In [ ]:
loss_small = loss[loss["Country Name"].isin(df_master["country"])].copy()

year_cols = ["2022", "2023", "2024"]

loss_small[year_cols] = loss_small[year_cols].apply(pd.to_numeric, errors="coerce")

loss_small["t_d_losses_pct"] = loss_small[year_cols].mean(axis=1)

loss_avg = loss_small[["Country Name", "t_d_losses_pct"]] \
    .rename(columns={"Country Name": "country"})

loss_avg


,country,t_d_losses_pct
14,Austria,4.427814
17,Belgium,3.931291
55,Germany,4.790575
58,Denmark,5.634087
70,Spain,8.575117
75,Finland,3.731238
77,France,7.198169
111,Ireland,7.522936
116,Italy,6.818453
176,Netherlands,4.074141


In [32]:
df_master = df_master.merge(loss_avg, on="country", how="left")
df_master


,country,elec_price_eur_kwh,carbon_intensity_gco2_kwh,renew_share_pct,avg_temp_c,t_d_losses_pct
0,Ireland,0.040593,296.400000,15.863333,10.463333,7.522936
1,Germany,0.047116,374.376667,125.048333,10.956667,4.790575
2,Netherlands,0.045444,281.423333,34.255833,11.706667,4.074141
3,France,0.037833,57.543333,58.116667,12.950000,7.198169
4,Spain,0.032659,177.430000,69.998333,15.140000,8.575117
5,Italy,0.048737,327.526667,58.158333,14.476667,6.818453
6,Belgium,0.042641,133.650000,18.480000,11.890000,3.931291
7,Poland,0.045631,666.380000,25.310000,10.300000,5.989562
8,Austria,0.035041,118.426667,42.835833,8.916667,4.427814
9,Denmark,0.038308,162.113333,32.487500,9.770000,5.634087


In [33]:
df_master["t_d_losses_pct"].describe()
df_master[df_master["t_d_losses_pct"].isna()][["country"]]


,country


In [34]:
df_master.to_csv("data_processed/country_indicators_2022_2024.csv", index=False)


In [ ]:
df = df_master.copy()

cols_lower_better = ["elec_price_eur_kwh", "carbon_intensity_gco2_kwh", "avg_temp_c", "t_d_losses_pct"]
cols_higher_better = ["renew_share_pct"]

def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

for c in cols_lower_better:
    df[c + "_norm"] = 1 - minmax(df[c])   # invert because lower is better

for c in cols_higher_better:
    df[c + "_norm"] = minmax(df[c])

norm_cols = [c + "_norm" for c in cols_lower_better + cols_higher_better]

df[["country"] + norm_cols]


,country,elec_price_eur_kwh_norm,carbon_intensity_gco2_kwh_norm,avg_temp_c_norm,t_d_losses_pct_norm,renew_share_pct_norm
0,Ireland,0.273236,0.581647,0.377963,0.217219,0.000000
1,Germany,0.054384,0.459060,0.338093,0.781304,1.000000
2,Netherlands,0.110462,0.605192,0.277478,0.929209,0.168453
3,France,0.365805,0.957155,0.176994,0.284266,0.386988
4,Spain,0.539389,0.768681,0.000000,0.000000,0.495810
5,Italy,0.000000,0.532713,0.053610,0.362656,0.387370
6,Belgium,0.204523,0.837507,0.262662,0.958700,0.023965
7,Poland,0.104211,0.000000,0.391164,0.533778,0.086520
8,Austria,0.459493,0.861440,0.502963,0.856195,0.247035
9,Denmark,0.349870,0.792760,0.433998,0.607164,0.152257


In [36]:
scenarios = {
    "Balanced": {
        "elec_price_eur_kwh_norm": 0.20,
        "carbon_intensity_gco2_kwh_norm": 0.20,
        "renew_share_pct_norm": 0.20,
        "avg_temp_c_norm": 0.20,
        "t_d_losses_pct_norm": 0.20
    },
    "Cost-focused": {
        "elec_price_eur_kwh_norm": 0.40,
        "carbon_intensity_gco2_kwh_norm": 0.15,
        "renew_share_pct_norm": 0.10,
        "avg_temp_c_norm": 0.15,
        "t_d_losses_pct_norm": 0.20
    },
    "Sustainability-focused": {
        "elec_price_eur_kwh_norm": 0.15,
        "carbon_intensity_gco2_kwh_norm": 0.35,
        "renew_share_pct_norm": 0.30,
        "avg_temp_c_norm": 0.10,
        "t_d_losses_pct_norm": 0.10
    },
    "Reliability-focused": {
        "elec_price_eur_kwh_norm": 0.20,
        "carbon_intensity_gco2_kwh_norm": 0.10,
        "renew_share_pct_norm": 0.10,
        "avg_temp_c_norm": 0.10,
        "t_d_losses_pct_norm": 0.50
    }
}

def score_scenario(df_norm, weights: dict):
    w = pd.Series(weights)
    return (df_norm[w.index] * w).sum(axis=1)

results = []
for name, weights in scenarios.items():
    tmp = df[["country"] + list(weights.keys())].copy()
    tmp["scenario"] = name
    tmp["score"] = score_scenario(df, weights)
    results.append(tmp[["country", "scenario", "score"]])

scores = pd.concat(results).sort_values(["scenario", "score"], ascending=[True, False])
scores.head(20)


,country,scenario,score
12,Norway,Balanced,0.834252
11,Finland,Balanced,0.795296
10,Sweden,Balanced,0.751069
8,Austria,Balanced,0.585425
1,Germany,Balanced,0.526568
9,Denmark,Balanced,0.467210
6,Belgium,Balanced,0.457471
3,France,Balanced,0.434242
2,Netherlands,Balanced,0.418159
4,Spain,Balanced,0.360776


In [ ]:
import numpy as np
import pandas as pd

norm_cols = [
    "elec_price_eur_kwh_norm",
    "carbon_intensity_gco2_kwh_norm",
    "renew_share_pct_norm",
    "avg_temp_c_norm",
    "t_d_losses_pct_norm"
]

X = df[norm_cols].copy()

eps = 1e-12

P = X / (X.sum(axis=0) + eps)

n = len(X)
k = 1 / np.log(n)

entropy = -k * (P * np.log(P + eps)).sum(axis=0)

d = 1 - entropy

w_entropy = d / d.sum()

w_entropy = w_entropy.sort_values(ascending=False)
w_entropy


renew_share_pct_norm              0.288642
elec_price_eur_kwh_norm           0.270358
avg_temp_c_norm                   0.223994
t_d_losses_pct_norm               0.131167
carbon_intensity_gco2_kwh_norm    0.085839
dtype: float64

In [38]:
df["entropy_score"] = (df[w_entropy.index] * w_entropy.values).sum(axis=1)

entropy_ranking = df[["country", "entropy_score"]].sort_values("entropy_score", ascending=False)
entropy_ranking


,country,entropy_score
12,Norway,0.793221
11,Finland,0.726999
10,Sweden,0.714737
1,Germany,0.520962
8,Austria,0.494443
9,Denmark,0.383440
3,France,0.369693
4,Spain,0.354923
6,Belgium,0.318687
2,Netherlands,0.314471


In [ ]:
scenario_wide = scores.pivot(index="country", columns="scenario", values="score")

scenario_wide["Entropy"] = df.set_index("country")["entropy_score"]

rank_wide = scenario_wide.rank(ascending=False)

rank_corr = rank_wide.corr(method="spearman").round(3)
rank_corr


scenario,Balanced,Cost-focused,Reliability-focused,Sustainability-focused,Entropy
scenario,,,,,
Balanced,1.000,0.951,0.879,0.923,0.967
Cost-focused,0.951,1.000,0.846,0.857,0.912
Reliability-focused,0.879,0.846,1.000,0.720,0.758
Sustainability-focused,0.923,0.857,0.720,1.000,0.973
Entropy,0.967,0.912,0.758,0.973,1.000


In [40]:
w_entropy.sum()


np.float64(1.0)

In [41]:
w_entropy.isna().any()


np.False_

In [ ]:
import numpy as np

def run_sensitivity(df, base_weights, n_sim=1000, perturb=0.15):
    """
    df: dataframe with normalised columns
    base_weights: dict of weights
    n_sim: number of simulations
    perturb: +/- percentage variation
    """
    
    cols = list(base_weights.keys())
    base = np.array(list(base_weights.values()))
    
    top3_counts = {country: 0 for country in df["country"]}
    
    for _ in range(n_sim):
        noise = np.random.uniform(1-perturb, 1+perturb, size=len(base))
        w = base * noise
        w = w / w.sum()
        
        scores = (df[cols] * w).sum(axis=1)
        ranked = df.assign(score=scores).sort_values("score", ascending=False)
        
        top3 = ranked.head(3)["country"].tolist()
        
        for c in top3:
            top3_counts[c] += 1
    
    sensitivity_df = pd.DataFrame({
        "country": list(top3_counts.keys()),
        "top3_frequency": list(top3_counts.values())
    })
    
    sensitivity_df["top3_percent"] = 100 * sensitivity_df["top3_frequency"] / n_sim
    
    return sensitivity_df.sort_values("top3_percent", ascending=False)


In [44]:
sens_balanced_30 = run_sensitivity(df, balanced_weights, n_sim=2000, perturb=0.30)
sens_balanced_30


,country,top3_frequency,top3_percent
11,Finland,2000,100.0
12,Norway,2000,100.0
10,Sweden,2000,100.0
1,Germany,0,0.0
0,Ireland,0,0.0
4,Spain,0,0.0
3,France,0,0.0
2,Netherlands,0,0.0
5,Italy,0,0.0
8,Austria,0,0.0


In [45]:
df[["country"] + norm_cols].assign(
    balanced_score=(df[[c+"_norm" for c in [
        "elec_price_eur_kwh",
        "carbon_intensity_gco2_kwh",
        "renew_share_pct",
        "avg_temp_c",
        "t_d_losses_pct"
    ]]] * 0.2).sum(axis=1)
).sort_values("balanced_score", ascending=False)


,country,elec_price_eur_kwh_norm,carbon_intensity_gco2_kwh_norm,renew_share_pct_norm,avg_temp_c_norm,t_d_losses_pct_norm,balanced_score
12,Norway,0.836022,1.000000,0.522134,1.000000,0.813107,0.834252
11,Finland,1.000000,0.901681,0.113324,0.961476,1.000000,0.795296
10,Sweden,0.872950,0.987680,0.384332,0.914601,0.595781,0.751069
8,Austria,0.459493,0.861440,0.247035,0.502963,0.856195,0.585425
1,Germany,0.054384,0.459060,1.000000,0.338093,0.781304,0.526568
9,Denmark,0.349870,0.792760,0.152257,0.433998,0.607164,0.467210
6,Belgium,0.204523,0.837507,0.023965,0.262662,0.958700,0.457471
3,France,0.365805,0.957155,0.386988,0.176994,0.284266,0.434242
2,Netherlands,0.110462,0.605192,0.168453,0.277478,0.929209,0.418159
4,Spain,0.539389,0.768681,0.495810,0.000000,0.000000,0.360776


In [46]:
balanced_rank = df[["country"] + norm_cols].assign(
    balanced_score=(df[norm_cols] * 0.2).sum(axis=1)
).sort_values("balanced_score", ascending=False)

balanced_rank


,country,elec_price_eur_kwh_norm,carbon_intensity_gco2_kwh_norm,renew_share_pct_norm,avg_temp_c_norm,t_d_losses_pct_norm,balanced_score
12,Norway,0.836022,1.000000,0.522134,1.000000,0.813107,0.834252
11,Finland,1.000000,0.901681,0.113324,0.961476,1.000000,0.795296
10,Sweden,0.872950,0.987680,0.384332,0.914601,0.595781,0.751069
8,Austria,0.459493,0.861440,0.247035,0.502963,0.856195,0.585425
1,Germany,0.054384,0.459060,1.000000,0.338093,0.781304,0.526568
9,Denmark,0.349870,0.792760,0.152257,0.433998,0.607164,0.467210
6,Belgium,0.204523,0.837507,0.023965,0.262662,0.958700,0.457471
3,France,0.365805,0.957155,0.386988,0.176994,0.284266,0.434242
2,Netherlands,0.110462,0.605192,0.168453,0.277478,0.929209,0.418159
4,Spain,0.539389,0.768681,0.495810,0.000000,0.000000,0.360776


In [47]:
sens_balanced_40 = run_sensitivity(df, scenarios["Balanced"], n_sim=2000, perturb=0.40)
sens_balanced_40


,country,top3_frequency,top3_percent
11,Finland,2000,100.0
12,Norway,2000,100.0
10,Sweden,2000,100.0
1,Germany,0,0.0
0,Ireland,0,0.0
4,Spain,0,0.0
3,France,0,0.0
2,Netherlands,0,0.0
5,Italy,0,0.0
8,Austria,0,0.0
